Домашку будет легче делать в колабе (убедитесь, что у вас runtype с gpu).

# Задание 1 (3 балла)

Обучите word2vec модели с негативным семплированием (cbow и skip-gram) аналогично тому, как это было сделано в семинаре. Вам нужно изменить следующие пункты: 
1) добавьте лемматизацию в предобработку (любым способом)  
2) измените размер окна в большую или меньшую сторону
3) измените размерность итоговых векторов

Выберете несколько не похожих по смыслу слов (не таких как в семинаре), и протестируйте полученные эмбединги (найдите ближайшие слова и оцените качество, как в семинаре). 
Постарайтесь обучать модели как можно дольше и на как можно большем количестве данных. (Но если у вас мало времени или ресурсов, то допустимо взять поменьше данных и поставить меньше эпох)

In [3]:
import numpy as np
import pandas as pd
from string import punctuation
from sklearn.model_selection import train_test_split
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_distances
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import re
import random
from tqdm.notebook import tqdm 
import pymorphy3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Torch version:", torch.__version__)

c:\Users\schig\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.4)
  from scipy.sparse import csr_matrix, issparse


Using device: cpu
Torch version: 2.8.0+cpu


In [6]:
morph = pymorphy3.MorphAnalyzer()

def preprocess(text):

    text = text.lower()
    tokens = re.sub(r'[^а-яa-zё]+', ' ', text).split()
    tokens = [morph.parse(token)[0].normal_form for token in tokens]
    return tokens

wiki = open("C:/Users/schig/Downloads/wiki_data_.txt", encoding="utf-8").read().split('\n')

In [7]:
vocab = Counter()

for text in wiki:
    vocab.update(preprocess(text))

filtered_vocab = {w for w,c in vocab.items() if c > 30}
len(filtered_vocab)
print(f'Количество слов в словаре: {len(filtered_vocab)}')

Количество слов в словре: 12448


In [8]:
word2id = {'PAD': 0}
for w in filtered_vocab:
    word2id[w] = len(word2id)

id2word = {i:w for w,i in word2id.items()}

sentences = []

for text in wiki:
    ids = [word2id[t] for t in preprocess(text) if t in word2id]
    if ids:
        sentences.append(ids)

In [ ]:
window = 2 #Я решила уменьшить окно
embedding_dim = 100
neg_samples = 10

def generate_skipgram(sentences, neg_samples=5):
    X, y, y_neg = [], [], []

    vocab_size = len(word2id)
    words_list = list(range(vocab_size))

    for sent in sentences:
        for i, target in enumerate(sent):
            context = sent[max(0, i-window):i] + sent[i+1:i+window+1]

            for ctx in context:
                X.append(target)
                y.append(ctx)

                negatives = np.random.choice(words_list, neg_samples)
                y_neg.append(negatives)

    return np.array(X), np.array(y), np.array(y_neg)

X_sg, y_sg, neg_sg = generate_skipgram(sentences[:1000])

In [13]:
def generate_cbow(sentences, neg_samples=10):
    X, y, y_neg = [], [], []

    vocab_size = len(word2id)
    words_list = list(range(vocab_size))

    for sent in sentences:
        for i, target in enumerate(sent):
            context = sent[max(0, i-window):i] + sent[i+1:i+window+1]
            if not context:
                continue

            X.append(context)
            y.append(target)

            negatives = np.random.choice(words_list, neg_samples)
            y_neg.append(negatives)

    return X, np.array(y), np.array(y_neg)

X_cb, y_cb, neg_cb = generate_cbow(sentences[:1000])

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Word2VecNeg(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.in_emb = nn.Embedding(vocab_size, emb_dim)
        self.out_emb = nn.Embedding(vocab_size, emb_dim)

    def forward(self, target, context, negatives):
        target_emb = self.in_emb(target) 
        ctx_emb = self.out_emb(context)           
        neg_emb = self.out_emb(negatives)         

        pos_score = torch.sum(target_emb * ctx_emb, dim=1)
        pos_loss = F.logsigmoid(pos_score)

        neg_score = torch.bmm(neg_emb, target_emb.unsqueeze(2)).squeeze()
        neg_loss = torch.sum(F.logsigmoid(-neg_score), dim=1)

        return -(pos_loss + neg_loss).mean()

In [15]:
model_sg = Word2VecNeg(len(word2id), embedding_dim).to(device)
opt = torch.optim.Adam(model_sg.parameters(), lr=1e-3)

X_t = torch.tensor(X_sg, dtype=torch.long).to(device)
y_t = torch.tensor(y_sg, dtype=torch.long).to(device)
neg_t = torch.tensor(neg_sg, dtype=torch.long).to(device)

batch_size = 1024
epochs = 3

for ep in range(epochs):
    total_loss = 0
    for i in range(0, len(X_t), batch_size):
        t = X_t[i:i+batch_size]
        c = y_t[i:i+batch_size]
        n = neg_t[i:i+batch_size]

        opt.zero_grad()
        loss = model_sg(t, c, n)
        loss.backward()
        opt.step()

        total_loss += loss.item()
    print(f"Epoch {ep+1}: loss = {total_loss:.3f}")

Epoch 1: loss = 19212.849
Epoch 2: loss = 15161.363
Epoch 3: loss = 11998.552


In [16]:
embeddings = model_sg.in_emb.weight.detach().cpu().numpy()

from sklearn.metrics.pairwise import cosine_distances

def most_similar(word, top=10):
    if word not in word2id:
        return None
    wid = word2id[word]
    dists = cosine_distances(embeddings[wid].reshape(1,-1), embeddings)[0]
    idx = np.argsort(dists)[:top]
    return [id2word[i] for i in idx]


In [25]:
test_words = ["вера", "музыка", "собака", "школа", "университет"]

for w in test_words:
    print(w, "→", most_similar(w))

вера → ['вера', 'хант', 'логический', 'отказываться', 'врезаться', 'сущность', 'фиолетовый', 'добро', 'ода', 'адаптация']
музыка → ['музыка', 'конго', 'рай', 'бестселлер', 'анна', 'сатис', 'вновь', 'густо', 'ягода', 'усовершенствовать']
собака → ['собака', 'свердловск', 'офисный', 'лю', 'выраженный', 'десантник', 'миллиметр', 'canon', 'подчинить', 'ямагути']
школа → ['школа', 'штормовой', 'матвеев', 'проводник', 'нижегородский', 'известный', 'движение', 'состав', 'снизу', 'изучать']
университет → ['университет', 'здание', 'четырнадцатый', 'левченко', 'вяземский', 'очень', 'придавать', 'прикомандировать', 'объяснить', 'аида']


# Задание 2 (2 балла)

Обучите 1 word2vec и 1 fastext модель в gensim. В каждой из модели нужно задать все параметры, которые мы разбирали на семинаре. Заданные значения должны отличаться от дефолтных и от тех, что мы использовали на семинаре.

In [3]:
import gensim
from gensim.models import Word2Vec, FastText
import pymorphy2
import re
from string import punctuation
import pymorphy3

In [2]:
wiki = open("C:/Users/Ale03x/Downloads/wiki_data (1).txt", encoding="utf-8").read().split('\n')

In [7]:
morph = pymorphy3.MorphAnalyzer()

def preprocess(text):
    text = text.lower()
    tokens = re.sub(r'[^а-яa-zё]+', ' ', text).split()
    lemmas = [morph.parse(tok)[0].normal_form for tok in tokens]
    return lemmas

sentences = [preprocess(text) for text in wiki if text.strip()]

In [8]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=150,  
    window=7,           
    min_count=20,      
    workers=4,
    sg=1,             
    negative=10,       
    hs=0,
    sample=1e-4,    
    alpha=0.02,         
    min_alpha=0.0005,   
    epochs=10           
)

In [9]:
ft_model = FastText(
    sentences=sentences,
    vector_size=120,   
    window=5,
    min_count=15,
    workers=4,
    sg=0,             
    negative=15,
    min_n=3,         
    max_n=6,
    alpha=0.03,
    epochs=7
)

In [14]:
test_words = ["учёный", "земля", "техника", "наука", "город"]

for w in test_words:
    if w in w2v_model.wv:
        print(f"\nWord2Vec — {w}:")
        print(w2v_model.wv.most_similar(w, topn=10))

    if w in ft_model.wv:
        print(f"\nFastText — {w}:")
        print(ft_model.wv.most_similar(w, topn=10))



Word2Vec — учёный:
[('наука', 0.7186141610145569), ('библиограф', 0.6819056272506714), ('ботаник', 0.6623092889785767), ('географ', 0.6484785079956055), ('биолог', 0.6433659195899963), ('философ', 0.638626754283905), ('зоолог', 0.636762797832489), ('профессор', 0.6328406929969788), ('исследователь', 0.6323025822639465), ('востоковед', 0.6316745281219482)]

FastText — учёный:
[('исследователь', 0.7612062692642212), ('геолог', 0.7250497937202454), ('исследовательский', 0.7243570685386658), ('астрофизик', 0.7236354947090149), ('филологический', 0.7217941880226135), ('социолог', 0.715803325176239), ('теолог', 0.7103053331375122), ('астрономия', 0.7065330147743225), ('гинеколог', 0.6997464299201965), ('математик', 0.6969508528709412)]

Word2Vec — земля:
[('десятина', 0.5844707489013672), ('пахотный', 0.5822940468788147), ('надел', 0.5722975134849548), ('селище', 0.5632002353668213), ('владение', 0.5477217435836792), ('пустошь', 0.5426561832427979), ('саженый', 0.5316250920295715), ('стров'

# Задание 3 (3 балла)

Используя датасет для классификации (labeled.csv), обучите классификатор на базе эмбеддингов. Оцените качество на отложенной выборке.   
В качестве эмбеддинг модели вы можете использовать одну из моделей обученных в предыдущем задании или использовать одну из предобученных моделей с rusvectores (удостоверьтесь что правильно воспроизводите предобработку в этом случае!)  
Для того, чтобы построить эмбединг целого текста, усредните вектора отдельных слов в один общий вектор. 
В качестве алгоритма классификации используйте LogisicticRegression (можете попробовать SGDClassifier, чтобы было побыстрее)  
F1 мера должна быть выше 20%. 

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, roc_auc_score
import gensim
import re
import pymorphy3
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("C:/Users/schig/Downloads/labeled.csv", encoding='utf-8')
df = df.rename(columns={'comment': 'text', 'toxic': 'label'})

morph = pymorphy3.MorphAnalyzer()

def preprocess(text):
    text = str(text).lower()
    tokens = re.sub(r'[^а-яa-zё]+', ' ', text).split()
    lemmas = []
    for token in tokens:
        if len(token) > 2: 
            try:
                lemma = morph.parse(token)[0].normal_form
                lemmas.append(lemma)
            except:
                lemmas.append(token)
    return lemmas

tqdm.pandas(desc="Лемматизация")
df['tokens'] = df['text'].progress_apply(preprocess)
sentences = df['tokens'].tolist()

model_path = "word2vec_model_toxic.model"
w2v_model = gensim.models.Word2Vec(
        sentences=sentences,
        vector_size=200,     
        window=7,           
        min_count=2,       
        workers=4,
        epochs=15,        
        sg=1               
    )

w2v_model.save(model_path)

def text_to_embedding(tokens, model, vector_size=200):
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

X = np.array([text_to_embedding(tokens, w2v_model) for tokens in tqdm(df['tokens'], desc="Векторизация")])
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Классы в train: {np.bincount(y_train.astype(int))}")
print(f"Классы в test: {np.bincount(y_test.astype(int))}")

clf = LogisticRegression(
    max_iter=1000, 
    random_state=42,
    class_weight='balanced'  
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)[:, 1]

f1 = f1_score(y_test, y_pred, average='binary')  
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"F1-score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-toxic', 'Toxic']))

print("\nПример предсказания для тестовых комментариев:")
test_comments = [
    "Классно, спасибо!",
    "Закрой свой грязный рот!"
]

for comment in test_comments:
    tokens = preprocess(comment)
    embedding = text_to_embedding(tokens, w2v_model).reshape(1, -1)
    prediction = clf.predict(embedding)[0]
    probability = clf.predict_proba(embedding)[0, 1]
    label = "Токсичный" if prediction > 0.5 else "Нетоксичный"
    print(f"Комментарий: '{comment[:50]}...'")
    print(f"  Предсказание: {label} (вероятность: {probability:.2f})")
    print("-" * 50)

print("\nАнализ наиболее важных слов для классификации:")
coef = clf.coef_[0]
word_importance = {}
vocab = list(w2v_model.wv.key_to_index.keys())

for word in vocab[:100]: 
    if word in w2v_model.wv:
        word_vec = w2v_model.wv[word]
        importance = np.dot(word_vec, coef)
        word_importance[word] = importance

sorted_words = sorted(word_importance.items(), key=lambda x: x[1], reverse=True)
print("Топ-10 слов, способствующих токсичности:")
for word, imp in sorted_words[:10]:
    print(f"  {word}: {imp:.4f}")

print("\nТоп-10 слов, способствующих нетоксичности:")
for word, imp in sorted_words[-10:]:
    print(f"  {word}: {imp:.4f}")

Векторизация: 100%|██████████| 14412/14412 [00:00<00:00, 17601.99it/s]


Train size: (11529, 200), Test size: (2883, 200)
Классы в train: [7668 3861]
Классы в test: [1918  965]
F1-score: 0.8206
ROC-AUC: 0.9361

Classification Report:
              precision    recall  f1-score   support

   Non-toxic       0.92      0.89      0.90      1918
       Toxic       0.79      0.85      0.82       965

    accuracy                           0.88      2883
   macro avg       0.86      0.87      0.86      2883
weighted avg       0.88      0.88      0.88      2883


Пример предсказания для тестовых комментариев:
Комментарий: 'Классно, спасибо!...'
  Предсказание: Нетоксичный (вероятность: 0.01)
--------------------------------------------------
Комментарий: 'Закрой свой грязный рот!...'
  Предсказание: Токсичный (вероятность: 1.00)
--------------------------------------------------

Анализ наиболее важных слов для классификации:
Топ-10 слов, способствующих токсичности:
  ты: 9.0592
  себя: 4.8493
  вы: 3.5984
  тут: 2.0193
  свой: 0.9597
  он: 0.0024
  этот: -0.9755
 

# Задание 4 (2 доп балла)

В тетрадку с фастекстом добавьте код для обучения с negative sampling (задача сводится к бинарной классификации) и обучите модель. Проверьте полученную модель на нескольких словах. Похожие слова должны быть похожими по смыслу и по форме.

In [10]:
import re
import random
import numpy as np
import torch
import torch.nn as nn
from string import punctuation
from collections import Counter
from sklearn.metrics.pairwise import cosine_distances

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

wiki = open("C:/Users/schig/Downloads/wiki_data_.txt", encoding="utf-8").read().split('\n')

def tokenize(text):
    tokens = re.sub('#+', ' ', text.lower()).split()
    tokens = [t.strip(punctuation) for t in tokens]
    return [t for t in tokens if t]

def ngrammer(raw_string, n):
    raw_string = '<' + raw_string + '>'
    return [raw_string[i:i+n] for i in range(len(raw_string)-n+1)
            if raw_string[i:i+n] not in ['<', '>']]

def split_tokens(tokens, min_n, max_n):
    res = []
    for t in tokens:
        subs = []
        for n in range(min_n, max_n+1):
            if len(t) > n:
                subs.extend(ngrammer(t, n))
        res.append(subs)
    return res

class SubwordTokenizer:
    def __init__(self, ngram_range=(2,4), min_count=10):
        self.min_n, self.max_n = ngram_range
        self.min_count = min_count

    def build_vocab(self, texts):
        fw, sw = Counter(), Counter()
        for text in texts:
            tokens = tokenize(text)
            fw.update(tokens)
            subs = split_tokens(tokens, self.min_n, self.max_n)
            for s in subs:
                sw.update(set(s))
        self.fullword_vocab = {w for w,c in fw.items() if c >= self.min_count}
        self.subword_vocab = {w for w,c in sw.items() if c >= self.min_count * 100}
        self.vocab = self.fullword_vocab | self.subword_vocab
        self.id2word = {i:w for i,w in enumerate(self.vocab)}
        self.word2id = {w:i for i,w in self.id2word.items()}

    def __call__(self, text):
        tokens = tokenize(text)
        subs = split_tokens(tokens, self.min_n, self.max_n)
        res = []
        for t, s in zip(tokens, subs):
            items = []
            if t in self.vocab:
                items.append(self.word2id[t])
            items += [self.word2id[x] for x in set(s) if x in self.word2id]
            if items:
                res.append(items)
        return res

def pad_sequences(seqs, maxlen):
    arr = np.zeros((len(seqs), maxlen), dtype='int64')
    for i,s in enumerate(seqs):
        arr[i,:min(len(s),maxlen)] = s[:maxlen]
    return arr

def sample_negatives(vocab_size, pos, k):
    res = []
    while len(res) < k:
        n = random.randint(0, vocab_size-1)
        if n != pos:
            res.append(n)
    return res

def gen_batches(sentences, tokenizer, window=10, batch_size=512, maxlen=20, negs=5):
    vocab_size = len(tokenizer.vocab)
    left = (window/2).__ceil__()
    right = window//2
    X,y,l = [],[],[]
    while True:
        for sent in sentences:
            enc = tokenizer(sent)
            for i in range(len(enc)):
                ctx = enc[max(0,i-left):i] + enc[i+1:i+right+1]
                for c in ctx:
                    pos = c[0]
                    X.append(enc[i])
                    y.append(pos)
                    l.append(1)
                    for n in sample_negatives(vocab_size, pos, negs):
                        X.append(enc[i])
                        y.append(n)
                        l.append(0)
                    if len(X) >= batch_size:
                        yield (
                            torch.LongTensor(pad_sequences(X, maxlen)).to(device),
                            torch.LongTensor(y).to(device),
                            torch.FloatTensor(l).to(device)
                        )
                        X,y,l = [],[],[]

class FastTextNS(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.in_emb = nn.Embedding(vocab_size, dim)
        self.out_emb = nn.Embedding(vocab_size, dim)

    def forward(self, x, t):
        v = self.in_emb(x).mean(1)
        u = self.out_emb(t)
        return (v*u).sum(1)

tokenizer = SubwordTokenizer()
tokenizer.build_vocab(wiki)

model = FastTextNS(len(tokenizer.vocab), 100).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

gen = gen_batches(wiki[:19000], tokenizer)

for epoch in range(5):
    model.train()
    s = 0
    for _ in range(5000):
        X,y,l = next(gen)
        opt.zero_grad()
        loss = loss_fn(model(X,y), l)
        loss.backward()
        opt.step()
        s += loss.item()
    print(epoch, s/5000)

emb = model.in_emb.weight.detach().cpu().numpy()
id2word = list(tokenizer.fullword_vocab)
fw_emb = np.zeros((len(id2word), emb.shape[1]))

for i,w in enumerate(id2word):
    idxs = tokenizer(w)[0]
    fw_emb[i] = emb[idxs].mean(0)

def most_similar(word, k=20):
    idxs = tokenizer(word)[0]
    v = emb[idxs].mean(0)
    d = cosine_distances(v.reshape(1,-1), fw_emb)[0]
    return [id2word[i] for i in d.argsort()[:k]]

print(most_similar("семья"))
print(most_similar("университет"))
print(most_similar("делать"))


0 0.8934022015392781
1 0.3627844838306308
2 0.3065386827833951
3 0.3048834515064955
4 0.29785709234178065
['семья', 'семьёй', 'семьях', 'семьи»', 'семьям', 'семьями', 'семь', 'семью', 'семьи', 'дарья', 'angel', 'обладает', 'русскую', '«красные', 'спорта', 'сиденья', 'большие', 'таллине', 'обладателем', 'сохраняла']
['университет', 'университета', 'университетом', 'университеты', 'университету', 'университетах', 'университете', 'университетский', 'университетских', 'университетского', 'университетов', 'университетском', 'университетские', 'университетской', 'универсиады', 'универсиаде', 'универсал', 'универсальный', 'универсальным', 'универсалы']
['делать', 'проделать', 'делам', 'делался', 'делах', 'делают', 'делает', 'делал', 'делались', 'переделать', 'собирать', 'делился', 'дела»', 'сделать', 'посылать', 'делали', 'прислать', 'делалось', 'собирают', 'делало']
